# Summary Buffer Memory
> Progressively summarize older conversation turns to fit more history in fewer tokens.

`SummaryBufferMemory` keeps recent turns verbatim and compresses older turns
into a running summary using a caller-supplied `compact_fn`.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.memory import SummaryBufferMemory

## Define a Compaction Function
In production you would call an LLM here. For this recipe we use a simple mock.

In [ ]:
def mock_compact(turns):
    """Concatenate turn previews as a stand-in for LLM summarization."""
    return "Summary: " + " | ".join(t.content[:20] for t in turns)

print("Compaction function ready.")

## Create the Memory
`summary_priority` controls where the summary appears in the context window
(lower = closer to the system prompt).

In [ ]:
memory = SummaryBufferMemory(
    max_tokens=1024,
    compact_fn=mock_compact,
    summary_priority=6,
)

print(f"Max tokens: {memory.max_tokens}")

## Add Messages Until Compaction Triggers
We add enough messages to push the memory over its token limit, forcing
the buffer to compact older turns into a summary.

In [ ]:
messages = [
    ("user", "Tell me about the history of Python."),
    ("assistant", "Python was created by Guido van Rossum and released in 1991. "
     "It emphasizes code readability and allows programmers to express concepts "
     "in fewer lines of code than languages like C++ or Java."),
    ("user", "What about its name? Why Python?"),
    ("assistant", "The name comes from Monty Python's Flying Circus, the BBC "
     "comedy series. Guido was a fan of the show and wanted a short, unique, "
     "and slightly mysterious name."),
    ("user", "What version is current?"),
    ("assistant", "Python 3.12 is the latest stable release. Python 2 reached "
     "end of life in January 2020."),
    ("user", "Tell me about type hints."),
    ("assistant", "Type hints were introduced in PEP 484. They let you annotate "
     "function signatures and variables with expected types without changing "
     "runtime behavior."),
    ("user", "And async support?"),
    ("assistant", "Python added async/await syntax in 3.5 via PEP 492. The "
     "asyncio library provides the event loop infrastructure."),
]

for role, content in messages:
    memory.add_message(role, content)

print(f"Messages added: {len(messages)}")

## Retrieve Context Items
The returned items include both the summary and the recent verbatim turns.

In [ ]:
items = memory.as_context_items()

print(f"Context items: {len(items)}\n")
for item in items:
    preview = str(item.content)[:80]
    print(f"  [{item.source_type.value}] {preview}...")

## Key Takeaways
- `SummaryBufferMemory` blends verbatim recent turns with a compressed summary
- Supply a `compact_fn` (typically an LLM call) to control how summaries are made
- `summary_priority` determines where the summary is placed in the context window
- This strategy retains more conversational history than a pure sliding window

**Next:** [Graph Memory](03_graph_memory.ipynb)